# Heterogeneous Beliefs Model of Stock Market Predictability

## Description
This notebook implements the strategy described in the paper "Heterogeneous Beliefs Model of Stock Market Predictability" by Jiho Park, published on 2024-06-12 ([arXiv:2406.08448](https://arxiv.org/abs/2406.08448)).

The strategy is based on a model of heterogeneous beliefs where investors have different perceptions of an asset's fundamental value and stochastic supply. This leads to momentum and reversal patterns in stock prices. The model predicts co-movement, lead-lag effects, and cross-sectional momentum and reversal.

## Phase 1 — Trading Context & Objectives
In this phase, we will configure the trading universe, parameters, and define our hypothesis.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

### Configuration
Here we define the trading universe, parameters, and our hypothesis.

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
LOOKBACK_PERIOD = 252
MOMENTUM_PERIOD = 21
REVERSAL_PERIOD = 63
POSITION_SIZE = 0.02

# Hypothesis
"""
We hypothesize that stocks exhibiting strong momentum over the past 21 days will continue to trend,
while stocks showing significant reversal over the past 63 days will revert to their mean.
"""


## Phase 2 — Data Download & Feature Computation

### Data Download
In this phase, we download historical price data for the defined universe and compute the momentum and reversal signals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download historical data
data = yf.download(UNIVERSE, start='2010-01-01', end='2024-06-12', group_by='ticker')
prices = pd.DataFrame({ticker: data[ticker]['Adj Close'] for ticker in UNIVERSE})

# Compute momentum and reversal signals
momentum = prices.pct_change(periods=MOMENTUM_PERIOD).shift(1)
reversal = prices.pct_change(periods=REVERSAL_PERIOD).shift(1)

# Cross-sectional normalization
momentum_zscore = (momentum - momentum.mean()) / momentum.std()
reversal_zscore = (reversal - reversal.mean()) / reversal.std()


## Phase 3 — Signal Generation & Portfolio Construction

### Signal Generation
Here we generate trading signals based on the momentum and reversal z-scores.

In [ ]:
# Generate trading signals
signals = momentum_zscore * reversal_zscore
signals = signals.rank(axis=1, pct=True)

# Position sizing and portfolio construction
positions = signals.mul(POSITION_SIZE)
positions = positions.sub(positions.sum(axis=1), axis=0)


## Phase 4 — Vectorized Backtest

### Backtest
In this phase, we perform a vectorized backtest of the strategy, ensuring no look-ahead bias.

In [ ]:
# Vectorized backtest
returns = prices.pct_change().shift(-1)
strategy_returns = (returns * positions).sum(axis=1)
cumulative_returns = (1 + strategy_returns).cumprod()


## Phase 5 — Performance Metrics

### Performance Evaluation
Here we calculate performance metrics such as Sharpe, Sortino, Calmar ratios, max drawdown, and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
annual_return = strategy_returns.mean() * 252
annual_volatility = strategy_returns.std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / (strategy_returns[strategy_returns < 0].std() * np.sqrt(252))
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar_ratio = annual_return / abs(max_drawdown)

# Plot equity curve
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2f}')


## Phase 6 — Monitoring Stub

### Monitoring
This phase includes a function that prints daily P&L and current positions given live data.

In [ ]:
# Monitoring stub
def monitor_portfolio(prices, positions):
    daily_pnl = (prices.pct_change() * positions).sum(axis=1)
    current_positions = positions.iloc[-1]
    print(f'Daily P&L: {daily_pnl.iloc[-1]:.2f}')
    print('Current Positions:')
    print(current_positions)

# Example usage
monitor_portfolio(prices, positions)
